# Intro 2: the agent loop and tool calling

In the experiments a language model gets a question about Hetionet and has to answer it with data from the graph. The model cannot reach the database itself. What it can do is ask us to run a Cypher query for it, and this mechanism is called tool calling. Our loop around it is in `src/agent.py` and uses the `openai` package, which works with any provider that offers the same API.

Below we look at what the model receives and then replay one run from our pilot log step by step. Nothing in this notebook calls a model, so it runs without an API key.

In [1]:
import os, sys

if not os.path.exists("pyproject.toml"):
    os.chdir("../..")
sys.path.append("src")

## What the model receives

Every request carries the system prompt, the question and the description of our one tool. The description is plain JSON with the name of the tool, a sentence about what it does and the arguments it takes.

In [2]:
import json
from agent import TOOLS
from prompt import SYSTEM_PROMPT

print(json.dumps(TOOLS[0], indent=2))

{
  "type": "function",
  "function": {
    "name": "run_cypher",
    "description": "Run a read-only Cypher query (Neo4j 3.5) on the Hetionet graph. Returns columns, rows, total_rows, truncated, or an error message.",
    "parameters": {
      "type": "object",
      "properties": {
        "query": {
          "type": "string",
          "description": "The Cypher query"
        }
      },
      "required": [
        "query"
      ]
    }
  }
}


The system prompt is the same for every model. It describes the graph and warns about the old Cypher version we saw in the first notebook.

In [3]:
print(SYSTEM_PROMPT)

You answer questions about Hetionet, a biomedical knowledge graph stored in Neo4j 3.5.
Use the run_cypher tool to query it. Base every answer on query results, never on memory.

Cypher dialect: Neo4j 3.5. Do not use newer syntax such as EXISTS { ... } subqueries.
Queries are read-only. Results are capped at 50 rows; check the 'truncated' flag.

Node labels and key property (all have name and identifier):
Compound (1552), Disease (137), Gene (20945), Anatomy, BiologicalProcess, CellularComponent,
MolecularFunction, Pathway, PharmacologicClass, SideEffect, Symptom.
Disease names are lowercase (e.g. 'bile duct cancer'); compound names are capitalised.

Relationship types (direction matters):
TREATS_CtD, PALLIATES_CpD (Compound->Disease); BINDS_CbG, UPREGULATES_CuG, DOWNREGULATES_CdG (Compound->Gene);
ASSOCIATES_DaG, UPREGULATES_DuG, DOWNREGULATES_DdG (Disease->Gene); CAUSES_CcSE (Compound->SideEffect);
PRESENTS_DpS (Disease->Symptom); LOCALIZES_DlA (Disease->Anatomy); RESEMBLES_CrC, RESEM

## How the loop works

The model can reply in two ways. If it replies with normal text, we take that as the final answer and stop. If it replies with a tool call, which is just the name `run_cypher` and a query as JSON, we run the query, add the result to the conversation and ask the model again. A broken query is not a failure at this point, because the model gets the error message back and can correct itself. We allow at most six rounds per question.

All of this is about thirty lines in `run_agent`. Every model runs with temperature 0 and the same seed, so that repeated runs differ as little as the provider allows, and every query and result is written to a log.

## Replaying a run from the pilot

In the pilot we ran five models on five questions. The log has one line per run, and the table shows how many queries each run needed.

In [4]:
import pandas as pd

runs = [json.loads(line) for line in open("results/pilot_20261025_131041.jsonl", encoding="utf-8")]
pd.DataFrame(runs).pivot(index="model", columns="qid", values="n_steps")

qid,q01,q02,q03,q04,q05
model,,,,,
bielik-11b,1.0,1.0,1.0,NaN,1.0
deepseek-v4-flash,1.0,1.0,1.0,8.0,2.0
gpt-oss-120b,1.0,1.0,1.0,3.0,1.0
mistral-small-24b,1.0,1.0,NaN,2.0,1.0
qwen3.8-27b,1.0,1.0,2.0,5.0,2.0


Most runs were done after a single query. The two `NaN` cells are runs where the provider returned an error before the model could answer, which the runner logs as an infrastructure error and does not count as a wrong answer.

The interesting runs are those where something went wrong on the way, so let's take one whose first query failed.

In [5]:
run = next(r for r in runs if r.get("n_steps", 0) > 1 and "error" in r["steps"][0]["result"])
print(run["model"], "on", run["qid"])

for i, step in enumerate(run["steps"], 1):
    print(f"\nquery {i}: {step['cypher']}")
    result = step["result"]
    if "error" in result:
        print("-> error:", result["error"].splitlines()[0])
    else:
        print("->", result["total_rows"], "rows:", result["rows"])

print("\nfinal answer:", run["final_answer"])

mistral-small-24b on q04

query 1: MATCH (d:Disease {name: 'osteoporosis'})-[:ASSOCIATES_DaG]->(g:Gene)<-[:BINDS_CbG]-(c:Compound) WHERE NOT (c)-[:TREATS_CtD]->(d) WITH c, COUNT(g) AS shared_genes COUNT DISTINCT g ORDER BY shared_genes DESC, c.name ASC LIMIT 5 RETURN c.name AS compound, shared_genes
-> error: Invalid input 'U': expected 'n/N' (line 1, column 165 (offset: 164))

query 2: MATCH (d:Disease {name: 'osteoporosis'})-[:ASSOCIATES_DaG]->(g:Gene)<-[:BINDS_CbG]-(c:Compound) WHERE NOT (c)-[:TREATS_CtD]->(d) WITH c, COUNT(g) AS shared_genes ORDER BY shared_genes DESC, c.name ASC LIMIT 5 RETURN c.name AS compound, shared_genes
-> 5 rows: [{'compound': 'Danazol', 'shared_genes': 3}, {'compound': 'Diethylstilbestrol', 'shared_genes': 3}, {'compound': 'Estriol', 'shared_genes': 3}, {'compound': 'Estrone', 'shared_genes': 3}, {'compound': 'Levonorgestrel', 'shared_genes': 3}]

final answer: The 5 compounds that bind the most genes associated with osteoporosis but are not known to treat

The first query has `COUNT DISTINCT g` without brackets. The server rejects it, the model reads the message, and the second attempt is correct. The log also keeps how the run was scored against the reference result:

In [6]:
{key: run[key] for key in ["exact", "precision", "recall", "f1", "tokens", "latency_s"]}

{'exact': True,
 'precision': 1.0,
 'recall': 1.0,
 'f1': 1.0,
 'tokens': 3368,
 'latency_s': 12.17}

## Running it live

A live run needs `LLM_BASE_URL` and `LLM_API_KEY` in a `.env` file (see `.env.example`). After that it is one call, and what comes back has the same fields as the log lines above:

```python
from agent import run_agent
from models import MODELS

run_agent(MODELS["gpt-oss-120b"], "Which compounds are known to treat multiple sclerosis?")
```

For whole experiments we use `src/run_experiments.py`, which goes over models, questions and repetitions and writes the log.

The thing to remember for the rest of the project is that the model only ever writes queries. Our code runs them, and every query and every result ends up in the log, so we can compare models and repeated runs later without calling the models again.